Hour 1 — Why Use a Database?

Import DuckDB

In [2]:
import pandas as pd
import duckdb

Now create a connection:

In [3]:
con = duckdb.connect("../data/supply_chain.duckdb")

Let's load yesterday's pipeline output into Python.

In [4]:
orders_df = pd.read_csv("../data/day_06_pipeline_output.csv")

Create Your First Database Table

We'll execute SQL through our database connection:

In [5]:
con.execute("""
    CREATE OR REPLACE TABLE orders AS
    SELECT * FROM orders_df
""")

How can we verify that orders actually exists in the database?

In [6]:
con.execute("SHOW TABLES").df()

,name
0,orders


Query the Database

In [7]:
con.execute("""
    SELECT *
    FROM orders
""").df()

,order_id,product,quantity,price,order_date,total_sales
0,1001,Laptop,1.0,1200,2026-01-05,1200.0
1,1002,Monitor,2.0,300,2026-01-07,600.0
2,1003,Keyboard,1.0,100,2026-01-10,100.0
3,1004,Laptop,1.0,1200,2026-02-02,1200.0
4,1005,Monitor,3.0,300,2026-02-15,900.0


SQL Aggregation From the Database

How much total revenue and total quantity did each product generate?

In [8]:
con.execute("""
    SELECT
        product,
        SUM(total_sales) AS total_revenue,
        SUM(quantity) AS total_quantity
    FROM orders
    GROUP BY product
    ORDER BY total_revenue DESC
""").df()

,product,total_revenue,total_quantity
0,Laptop,2400.0,2.0
1,Monitor,1500.0,5.0
2,Keyboard,100.0,1.0


Let's store the SQL result in Python.

In [9]:
product_summary_df = con.execute("""
    SELECT
        product,
        SUM(total_sales) AS total_revenue,
        SUM(quantity) AS total_quantity
    FROM orders
    GROUP BY product
    ORDER BY total_revenue DESC
""").df()

Hour 2 — Tables, Schemas & SQL Pipeline Design

Create a Schema

In [10]:
con.execute("""
    CREATE SCHEMA IF NOT EXISTS analytics
""")

Create an Analytics Table

In [11]:
con.execute("""
    CREATE OR REPLACE TABLE analytics.product_summary AS

    SELECT
        product,
        SUM(total_sales) AS total_revenue,
        SUM(quantity) AS total_quantity
    FROM orders
    GROUP BY product
""")

Query the Schema Table

In [12]:
con.execute(""" 
    SELECT *
    FROM analytics.product_summary
    ORDER BY total_revenue DESC    
""").df()

,product,total_revenue,total_quantity
0,Laptop,2400.0,2.0
1,Monitor,1500.0,5.0
2,Keyboard,100.0,1.0


Create a Second Analytics Table

How much revenue did we generate by month?

In [13]:
con.execute("""
    SELECT *
    FROM orders
""").df()

,order_id,product,quantity,price,order_date,total_sales
0,1001,Laptop,1.0,1200,2026-01-05,1200.0
1,1002,Monitor,2.0,300,2026-01-07,600.0
2,1003,Keyboard,1.0,100,2026-01-10,100.0
3,1004,Laptop,1.0,1200,2026-02-02,1200.0
4,1005,Monitor,3.0,300,2026-02-15,900.0


In [15]:
con.execute("""
    CREATE OR REPLACE TABLE analytics.monthly_sales AS
    
    SELECT
        DATE_TRUNC('month',CAST(order_date AS DATE)) AS order_month,
        SUM(total_sales) AS total_revenue,
        COUNT(order_id) AS order_count
    FROM orders
    GROUP BY order_month
    ORDER BY order_month
""")

Let's verify analytics.monthly_sales

In [16]:
monthly_sales_df = con.execute("""
    SELECT *
    FROM analytics.monthly_sales
    ORDER BY order_month
""").df()

Hour 3 — Building Analytical Tables with SQL

In [17]:
con.execute("""
    CREATE OR REPLACE TABLE analytics.product_metrics AS

    SELECT 
        product,
        COUNT(order_id) AS order_count,
        SUM(quantity) AS total_quantity,
        SUM(total_sales) AS total_revenue,
        AVG(total_sales) AS average_order_value
    FROM orders
    GROUP BY product
""")

Rank Products

In [18]:
con.execute("""
    SELECT
        *,
        RANK() OVER(
            ORDER BY total_revenue DESC)
        AS revenue_rank
    FROM analytics.product_metrics
""").df()

,product,order_count,total_quantity,total_revenue,average_order_value,revenue_rank
0,Laptop,2,2.0,2400.0,1200.0,1
1,Monitor,2,5.0,1500.0,750.0,2
2,Keyboard,1,1.0,100.0,100.0,3


Store the Ranked Results

In [19]:
con.execute("""
    CREATE OR REPLACE TABLE analytics.product_rankings AS

    SELECT
        *,
        RANK() OVER(
            ORDER BY total_revenue DESC)
        AS revenue_rank
    FROM analytics.product_metrics
""")

Let's verify the whole analytics layer.

In [21]:
con.execute("SHOW TABLES").df()

,name
0,orders


In [23]:
con.execute("""
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema IN ('main','analytics')
    ORDER BY table_schema, table_name
""").df()

,table_schema,table_name
0,analytics,monthly_sales
1,analytics,product_metrics
2,analytics,product_rankings
3,analytics,product_summary
4,main,orders


Hour 4 — Build the End-to-End Database Pipeline

Database Load Function

In [25]:
def load_orders_to_database(df, con):
    con.execute("""
        CREATE OR REPLACE TABLE orders AS
        SELECT * FROM df
    """)

Create the Analytics Layer Function

In [24]:
def build_analytics_tables(con):
    con.execute("""
        CREATE SCHEMA IF NOT EXISTS analytics
    """)

    con.execute("""
        CREATE OR REPLACE TABLE analytics.product_metrics AS

        SELECT 
            product,
            COUNT(order_id) AS order_count,
            SUM(quantity) AS total_quantity,
            SUM(total_sales) AS total_revenue,
            AVG(total_sales) AS average_order_value
        FROM orders
        GROUP BY product
    """)

Build the Orchestrator

In [26]:
def run_database_pipeline(input_path, con):

    raw_df = extract_data(input_path)

    validate_data(raw_df)

    clean_df = clean_sales_data(raw_df)

    load_orders_to_database(clean_df, con)

    build_analytics_tables(con)

Now we need to actually run the pipeline.

In [28]:
print("extract_data:", "extract_data" in globals())
print("validate_data:", "validate_data" in globals())
print("clean_sales_data:", "clean_sales_data" in globals())
print("load_orders_to_database:", "load_orders_to_database" in globals())
print("build_analytics_tables:", "build_analytics_tables" in globals())

extract_data: False
validate_data: False
clean_sales_data: False
load_orders_to_database: True
build_analytics_tables: True


In [29]:
def extract_data(file_path):
    df = pd.read_csv(file_path)
    return df

In [30]:
def validate_data(df):
    duplicate_orders = df.duplicated(
        subset=["order_id"]
    ).sum()

    invalid_values = (
        (df["quantity"] <= 0) |
        (df["price"] <= 0)
    ).sum()

    missing_values = df[
        ["order_id", "product", "price", "order_date"]
    ].isna().sum().sum()

    if invalid_values > 0:
        raise ValueError("Invalid quantity or price detected")

    if duplicate_orders > 0:
        raise ValueError("Duplicate order IDs detected")

    if missing_values > 0:
        raise ValueError("Missing required values detected")

    return {
        "duplicate_orders": duplicate_orders,
        "invalid_values": invalid_values,
        "missing_values": missing_values
    }

In [31]:
def clean_sales_data(df):
    clean_df = df.copy()

    clean_df["quantity"] = clean_df["quantity"].fillna(1)
    clean_df["order_date"] = pd.to_datetime(clean_df["order_date"])
    clean_df["total_sales"] = (
        clean_df["price"] * clean_df["quantity"]
    )

    return clean_df

In [32]:
run_database_pipeline("../data/day_06_clean_orders.csv", con)

In [33]:
print("extract_data:", "extract_data" in globals())
print("validate_data:", "validate_data" in globals())
print("clean_sales_data:", "clean_sales_data" in globals())
print("load_orders_to_database:", "load_orders_to_database" in globals())
print("build_analytics_tables:", "build_analytics_tables" in globals())

extract_data: True
validate_data: True
clean_sales_data: True
load_orders_to_database: True
build_analytics_tables: True


In [34]:
con.execute("""
    SELECT *
    FROM analytics.product_metrics
    ORDER BY total_revenue DESC
""").df()

,product,order_count,total_quantity,total_revenue,average_order_value
0,Laptop,2,2.0,2400.0,1200.0
1,Monitor,2,5.0,1500.0,750.0
2,Keyboard,1,1.0,100.0,100.0
